In [1]:
from utils import State, Action
from collections import OrderedDict
from torch import tensor


In [ ]:
coeffs

In [6]:

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from utils import ImmutableState, State, Action, board_status, get_local_board_status, get_all_valid_actions, is_terminal, change_state, terminal_utility, invert, load_data
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

class UTTTModel(nn.Module):
    def __init__(self):
        super(UTTTModel, self).__init__()
        self.conv1 = nn.Conv2d(4, 16, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.conv4 = nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1)
        #self.conv4b = nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1)
        #self.conv4c = nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1)
        self.conv5 = nn.Conv2d(64, 32, kernel_size=3, stride=1, padding=1)
        self.fc1 = nn.Linear(32 * 9 * 9, 192)  # 32 channels from last conv layer
        self.fc2 = nn.Linear(192, 256) #168 #224
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, 64)
        self.fc5 = nn.Linear(64, 1)
        #self.relu = nn.ReLU()
        #self.prelu = nn.PReLU()
        self.lrelu = nn.LeakyReLU(0.01)
    
    def forward(self, x):
        x = x.view(-1, 4, 9, 9)  # Reshape to 9x9 grid
        x = self.lrelu(self.conv1(x))
        x = self.lrelu(self.conv2(x))
        x = self.lrelu(self.conv3(x))
        x = self.lrelu(self.conv4(x))
        x = self.lrelu(self.conv5(x))
        x = x.view(x.size(0), -1)  # Flatten for FC layers
        x = self.lrelu(self.fc1(x))
        x = self.lrelu(self.fc2(x))
        x = self.lrelu(self.fc3(x))
        x = self.lrelu(self.fc4(x))
        #return torch.sigmoid(self.fc5(x)) #output range (0,1)
        return torch.tanh(self.fc5(x))  # Output score in range (-1, 1)

def state_to_tensor(state):
    """
    Convert a 3x3x3x3 Ultimate Tic-Tac-Toe board state into a 4x9x9 tensor for the neural network.
    """
    board_3x3x3x3 = state.board.copy()

    # Convert 3x3x3x3 nested board into 9x9
    board_9x9 = np.zeros((9, 9), dtype=np.float32)
    
    for meta_row in range(3):
        for meta_col in range(3):
            for local_row in range(3):
                for local_col in range(3):
                    global_row = meta_row * 3 + local_row
                    global_col = meta_col * 3 + local_col
                    board_9x9[global_row][global_col] = board_3x3x3x3[meta_row][meta_col][local_row][local_col]

    # Normalize values: 0 stays 0, AI (1) stays 1, Opponent (2) becomes -1
    board_9x9[board_9x9 == 2] = -1

    # Convert to PyTorch tensor and add batch dimension
    board_tensor = torch.tensor(board_9x9, dtype=torch.float32).unsqueeze(0)  # Shape: (1, 9, 9)
    
    turn_tensor = torch.full((1,9,9), 1 if state.fill_num==1 else -1, dtype=torch.float32)
    
    action_9x9 = np.zeros((9,9), dtype=np.float32)
    valid_actions = get_all_valid_actions(state)
    for meta_row, meta_col, local_row, local_col in valid_actions:
        global_row = meta_row * 3 + local_row
        global_col = meta_col * 3 + local_col
        action_9x9[global_row][global_col] = 1
    
    action_tensor = torch.tensor(action_9x9, dtype=torch.float32).unsqueeze(0)
    
    outcome_9x9=np.zeros((9, 9), dtype=np.float32)
    lbs = get_local_board_status(board_3x3x3x3)  # 3x3 array
    for i in range(3):  # iterate over meta board rows
        for j in range(3):  # iterate over meta board cols
            local_status = lbs[i, j]
            if local_status == 1:
                fill_value = 1.0
            elif local_status == 2:
                fill_value = -1.0
            else:
                fill_value = 0.0

            # Fill the corresponding 3x3 block in the global 9x9 board
            row_start, row_end = i * 3, (i + 1) * 3
            col_start, col_end = j * 3, (j + 1) * 3
            outcome_9x9[row_start:row_end, col_start:col_end] = fill_value

    outcome_tensor = torch.tensor(outcome_9x9, dtype=torch.float32).unsqueeze(0)
    
    return torch.cat([turn_tensor,board_tensor,outcome_tensor,action_tensor],dim=0)

def evaluation(state, model):
    """
    Evaluates the board using the trained neural network.
    """
    if state.is_terminal(): #correct?
        return 2*state.terminal_utility()-1
    state_tensor = state_to_tensor(state).unsqueeze(0)  # Add batch dimension
    with torch.no_grad():
        return model(state_tensor).item()  # Get NN evaluation score
    
def minimax(model, state, depth, alpha, beta, maximizing=True):
    if depth == 0 or state.is_terminal():
        return evaluation(state, model), None
    
    best_action = None
    
    if maximizing:
        max_eval = -float("inf")
        for action in state.get_all_valid_actions():
            new_state = state.change_state(action)
            eval, _ = minimax(model, new_state, depth - 1, alpha, beta, False)
            if eval > max_eval:
                max_eval = eval
                best_action = action
            alpha = max(alpha, eval)
            if beta <= alpha:
                break
        return max_eval, best_action
    else:
        min_eval = float("inf")
        for action in state.get_all_valid_actions():
            new_state = state.change_state(action)
            eval, _ = minimax(model, new_state, depth - 1, alpha, beta, True)
            if eval < min_eval:
                min_eval = eval
                best_action = action
            beta = min(beta, eval)
            if beta <= alpha:
                break
        return min_eval, best_action

class StudentAgent:
    def __init__(self):
        """Instantiates your agent.
        """
        self.model=UTTTModel()
        self.model.load_state_dict(coeffs)

    def choose_action(self, state: State) -> Action:
        """Returns a valid action to be played on the board.
        Assuming that you are filling in the board with number 1.

        Parameters
        ---------------
        state: The board to make a move on.
        """
        best_score, best_move = minimax(self.model, state, depth=3, alpha=float("-inf"), beta=float("inf"), maximizing=True)
        return best_move


In [7]:
model3=UTTTModel()
print(model3)
print(sum(p.numel() for p in model3.parameters()))

UTTTModel(
  (conv1): Conv2d(4, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv4): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv5): Conv2d(64, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=2592, out_features=192, bias=True)
  (fc2): Linear(in_features=192, out_features=256, bias=True)
  (fc3): Linear(in_features=256, out_features=128, bias=True)
  (fc4): Linear(in_features=128, out_features=64, bias=True)
  (fc5): Linear(in_features=64, out_features=1, bias=True)
  (lrelu): LeakyReLU(negative_slope=0.01)
)
667601


In [11]:
# Use this cell to test your agent in two full games against a random agent.
# The random agent will choose actions randomly among the valid actions.

class RandomStudentAgent(StudentAgent):
    def choose_action(self, state: State) -> Action:
        # If you're using an existing Player 1 agent, you may need to invert the state
        # to have it play as Player 2. Uncomment the next line to invert the state.
        # state = state.invert()

        # Choose a random valid action from the current game state
        return state.get_random_valid_action()

def run(your_agent: StudentAgent, opponent_agent: StudentAgent, start_num: int):
    your_agent_stats = {"timeout_count": 0, "invalid_count": 0}
    opponent_agent_stats = {"timeout_count": 0, "invalid_count": 0}
    turn_count = 0
    
    state = State(fill_num=start_num)
    
    while not state.is_terminal():
        turn_count += 1

        agent_name = "your_agent" if state.fill_num == 1 else "opponent_agent"
        agent = your_agent if state.fill_num == 1 else opponent_agent
        stats = your_agent_stats if state.fill_num == 1 else opponent_agent_stats

        start_time = time.time()
        action = agent.choose_action(state.clone())
        end_time = time.time()
        
        random_action = state.get_random_valid_action()
        if end_time - start_time > 3:
            print(f"{agent_name} timed out!")
            stats["timeout_count"] += 1
            action = random_action
        if not state.is_valid_action(action):
            print(f"{agent_name} made an invalid action!")
            stats["invalid_count"] += 1
            action = random_action
                
        state = state.change_state(action)

    print(f"== {your_agent.__class__.__name__} (1) vs {opponent_agent.__class__.__name__} (2) - First Player: {start_num} ==")
        
    if state.terminal_utility() == 1:
        print("You win!")
    elif state.terminal_utility() == 0:
        print("You lose!")
    else:
        print("Draw")

    for agent_name, stats in [("your_agent", your_agent_stats), ("opponent_agent", opponent_agent_stats)]:
        print(f"{agent_name} statistics:")
        print(f"Timeout count: {stats['timeout_count']}")
        print(f"Invalid count: {stats['invalid_count']}")
        
    print(f"Turn count: {turn_count}\n")

your_agent = lambda: StudentAgent()
opponent_agent = lambda: RandomStudentAgent() #StudentAgent(testmodel3,False) #RandomStudentAgent()

run(your_agent(), opponent_agent(), 1)
run(your_agent(), opponent_agent(), 2)

your_agent timed out!
== StudentAgent (1) vs RandomStudentAgent (2) - First Player: 1 ==
You win!
your_agent statistics:
Timeout count: 1
Invalid count: 0
opponent_agent statistics:
Timeout count: 0
Invalid count: 0
Turn count: 31

== StudentAgent (1) vs RandomStudentAgent (2) - First Player: 2 ==
You win!
your_agent statistics:
Timeout count: 0
Invalid count: 0
opponent_agent statistics:
Timeout count: 0
Invalid count: 0
Turn count: 34

